In [9]:
import pandas as pd
import os
import glob
from dateutil import parser

# Parameters
ENTRY_TIME = "09:30:00+05:30"
EXIT_TIME = "11:00:00+05:30"
VOL_LOOKBACK_DAYS = 1

In [10]:
def read_stock_data(file):
    df = pd.read_csv(file, parse_dates=["date"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df.sort_index()
    return df





def calculate_volatility(df, lookback=VOL_LOOKBACK_DAYS):
    # Calculate 5-min returns
    returns = df["open"].pct_change()

    # Extract just the date part for grouping, but keep as pandas Timestamp
    df = df.copy()
    df["date_only"] = df.index.floor("D")

    # Calculate daily std of 5-min returns
    daily_vol = returns.groupby(df["date_only"]).std()

    # Smooth using rolling average
    smoothed_vol = daily_vol.rolling(window=lookback).mean()

    return smoothed_vol




def predict_direction(df_day):
    try:
        row = df_day.between_time("09:30", "09:30").iloc[0]
        if row["RSI14"] < 30:
            return "sell"
        elif row["RSI14"] > 70:
            return "buy"
        else:
            return "hold"
    except:
        return "hold"



def simulate_trade(df_day, direction):
    # Filter time range between ENTRY_TIME and EXIT_TIME
    intraday = df_day.between_time(ENTRY_TIME, EXIT_TIME)

    if intraday.empty:
        return 0  # No data for this time range

    entry_price = intraday["open"].iloc[0]

    for idx, row in intraday.iterrows():
        current_price = row["open"]

        if direction == "buy":
            ret = (current_price - entry_price) / entry_price
        elif direction == "sell":
            ret = (entry_price - current_price) / entry_price
        else:
            return 0

        # Check for stop loss breach
        if ret < STOP_LOSS:
            print(f"Trade hit stop loss at {idx.time()}: {ret:.2%}")
            return STOP_LOSS

    # If no stop loss triggered, return final return at exit time
    exit_price = intraday["open"].iloc[-1]
    if direction == "buy":
        return (exit_price - entry_price) / entry_price
    else:  # direction == "sell"
        return (entry_price - exit_price) / entry_price







def process_day(date, all_stock_data):
    print(f"Processing date: {date}", flush=True)
    volatilities = {}

    for name, df in all_stock_data.items():
        try:
            df_filtered = df[df.index.date < date]
            if len(df_filtered) < VOL_LOOKBACK_DAYS * 78:  # 78 = 5-min bars/day
                continue
            vol_series = calculate_volatility(df_filtered)
            if not vol_series.empty:
                vol = vol_series.iloc[-1]
                volatilities[name] = vol
        except Exception as e:
            print(f"Error processing {name} on {date}: {e}", flush=True)
            continue

    if not volatilities:
        print(f"No volatility data for {date}", flush=True)
        return None

    # Sort stocks by descending volatility
    sorted_stocks = sorted(volatilities.items(), key=lambda x: x[1], reverse=True)[:3]


    # Try stocks in order of volatility
    for stock_name, vol in sorted_stocks:
        print(f"Trying stock: {stock_name} (Volatility: {vol:.4f})", flush=True)
        df = all_stock_data[stock_name]
        try:
            df_day = df[df.index.date == pd.to_datetime(date).date()]
            if df_day.empty:
                continue

            direction = predict_direction(df_day)
            if direction == "hold":
                continue  # Try next stock

            ret = simulate_trade(df_day, direction)

            return {
                "date": date,
                "stock": stock_name,
                "direction": direction,
                "return": ret,
                "latest_volatility": vol
            }

        except Exception as e:
            print(f"Simulation failed on {date} for {stock_name}: {e}", flush=True)
            continue

    # If no stock gave a buy/sell signal
    print(f"No actionable trade on {date}", flush=True)
    return {
                "date": date,
                "stock": "none",
                "direction": "none",
                "return": 0,
        "latest_volatility": None
            }


In [5]:

files = glob.glob("*_with_indicators_.csv")
all_stock_data = {os.path.basename(f).split(".")[0]: read_stock_data(f) for f in files}
all_dates = sorted(set.union(*[set(df.index.date) for df in all_stock_data.values()]))
start_date = all_dates[0]
end_date = all_dates[-1]

print("Start date:", start_date)
print("End date:", end_date)

Start date: 2015-02-02
End date: 2022-02-18


In [11]:
results = []
capital = 1000
target=1300
leverage = 4
tax_rate=0.0033#assuming invested 1 00 000,per tarde amount on avg
STOP_LOSS = -0.02#that is 2% of the stock price and 8% of our capital due to leverage
# print(len(all_dates))#1746
reinvested=0.9
traded_vols = []  # To collect volatility of only traded stocks

In [12]:
for date in all_dates[-365:]:
    res = process_day(date, all_stock_data)
    if res:

        # Store traded stock's volatility if a trade was made
        if res["stock"] != "none" and res["latest_volatility"] is not None:
            traded_vols.append(res["latest_volatility"])

        daily_return =(res["return"]* leverage * reinvested )
        capital *= (1 + daily_return )
        if daily_return != 0:
            capital -= capital * tax_rate
        res["capital"] = capital  # Track capital over time
        res["daily_leveraged_return"] = daily_return*100
        res["return"] = res["return"]*100# to convert to % in the results
        results.append(res)
      # # ✅ Stop if 30% return reached
      #   if capital >= target:
      #       print(f"🎯 Target reached on {date}. Capital: ₹{capital:.2f}")
      #       break
        if capital <0:
            print(f"you broke on {date}. Capital: ₹{capital:.2f}")
            break



results_df = pd.DataFrame(results)
results_df.to_csv("strategy_results.csv", index=False)
print("Backtest complete. Saved to strategy_results.csv")

if traded_vols:
    avg_traded_vol = sum(traded_vols) / len(traded_vols)
    print(f"📊 Average volatility of traded stocks: {avg_traded_vol:.6f}")
else:
    print("⚠️ No trades made. Cannot compute average traded volatility.")

Processing date: 2020-09-03
Trying stock: DLF_with_indicators_ (Volatility: 0.0050)
Trying stock: PEL_with_indicators_ (Volatility: 0.0044)
Trying stock: INDUSINDBK_with_indicators_ (Volatility: 0.0040)
No actionable trade on 2020-09-03
Processing date: 2020-09-04
Trying stock: VEDL_with_indicators_ (Volatility: 0.0044)
Trying stock: INDIGO_with_indicators_ (Volatility: 0.0041)
Trying stock: JINDALSTEL_with_indicators_ (Volatility: 0.0035)
Processing date: 2020-09-07
Trying stock: INDUSINDBK_with_indicators_ (Volatility: 0.0050)
Trying stock: INDIGO_with_indicators_ (Volatility: 0.0047)
Trying stock: DMART_with_indicators_ (Volatility: 0.0043)
No actionable trade on 2020-09-07
Processing date: 2020-09-08
Trying stock: BOSCHLTD_with_indicators_ (Volatility: 0.0035)
Trying stock: INDUSINDBK_with_indicators_ (Volatility: 0.0034)
Trying stock: JINDALSTEL_with_indicators_ (Volatility: 0.0033)
No actionable trade on 2020-09-08
Processing date: 2020-09-09
Trying stock: PEL_with_indicators_ (V

In [8]:
#1000.0 to 32219348.54001385 in 7 years 2015 to 2022 approx